# Assignment


## Brief

Write the Python codes for the following questions.


## Instructions

- Step 1: Run the followings cell to get connected to your MongoDB. Make sure your credential is in dotenv file.
- Step 2: Put your answer inside each function. Please do not construct your own function.
- Step 3: You can test your function under test section.
- Step 4. Run test my function to confirm if my code is working.


### Connections

In [1]:
import os

from dotenv import load_dotenv
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

In [2]:
# Load environment variables from .env file
load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI")
if not MONGODB_URI:
    raise ValueError(
        "❌ MONGODB_URI not found!\n"
        "Please create a .env file with your MongoDB credentials.\n"
        "See README.md for setup instructions."
    )
client = MongoClient(MONGODB_URI, server_api=ServerApi("1"))
# Send a ping to confirm a successful connection
try:
    client.admin.command("ping")
    print("✅ Successfully connected to MongoDB!")
except Exception as e:
    print(e)

✅ Successfully connected to MongoDB!


In [3]:
db = client.sample_mflix
movies = db.movies
print(f"📊 Database: {db.name}")
print(f"📁 Collection: movies ({movies.count_documents({})} documents)")

📊 Database: sample_mflix
📁 Collection: movies (21349 documents)



### Question 1

Question: From the `movies` collection, return the documents with the `plot` that starts with `"war"` in acending order of released date, print only title, plot and released fields. Limit the result to 5.


**Answer:**

In [5]:
filter_query = {
    "plot": {"$regex": "^war", "$options": "i"}  # case‑insensitive starts‑with
}

projection = {
    "_id": 0,          # exclude the default _id field    "title": 1,
    "plot": 1,
    "released": 1
}

cursor = (
    movies.find(filter_query, projection)
    .sort("released", 1)   # 1 = ascending order
    .limit(5)              # limit to 5 results
)

for movie in cursor:
    print(movie)

{'plot': 'Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.', 'released': datetime.datetime(1984, 3, 11, 0, 0)}
{'plot': 'Warrior/pacifist Princess Nausicaè desperately struggles to prevent two warring nations from destroying themselves and their dying planet.', 'released': datetime.datetime(1984, 3, 11, 0, 0)}
{'plot': 'Warlords Kagetora and Takeda each wish to prevent the other from gaining hegemony in feudal Japan. The two samurai leaders pursue one another across the countryside, engaging in massive ...', 'released': datetime.datetime(1991, 2, 8, 0, 0)}
{'plot': 'Warning! This synopsis contains spoilers Bajo las estrellas (beneath the stars) features the selfish...', 'released': datetime.datetime(2007, 6, 15, 0, 0)}
{'plot': 'Warring alien and predator races descend on a small town, where unsuspecting residents must band together for any chance of survival.', 'released': datetime.datetime(2007,

### Question 2

Question: Group by `rated` and count the number of movies in each.

**Answer:**

In [10]:
pipeline = [
    {
        "$group": {
            "_id": "$rated",
            "count": {"$sum": 1}
        }
    }
]

result = movies.aggregate(pipeline)

for doc in result:
    print(doc)

{'_id': 'TV-14', 'count': 89}
{'_id': 'TV-MA', 'count': 60}
{'_id': 'PG-13', 'count': 2321}
{'_id': 'GP', 'count': 44}
{'_id': 'R', 'count': 5537}
{'_id': 'TV-G', 'count': 59}
{'_id': 'Not Rated', 'count': 1}
{'_id': None, 'count': 9894}
{'_id': 'APPROVED', 'count': 709}
{'_id': 'PASSED', 'count': 181}
{'_id': 'TV-PG', 'count': 76}
{'_id': 'PG', 'count': 1852}
{'_id': 'G', 'count': 477}
{'_id': 'Approved', 'count': 5}
{'_id': 'M', 'count': 37}
{'_id': 'AO', 'count': 3}
{'_id': 'TV-Y7', 'count': 3}
{'_id': 'OPEN', 'count': 1}



### Question 3

Question: Count the number of movies with 3 comments or more.


**Answer:**


In [25]:
# Check how many movies have a `comments` field and its typical length
pipeline_check = [
    {
        "$project": {
            "commentCount": {
                "$size": {
                    "$ifNull": ["$comments", []]
                }
            }
        }
    },
    {
        "$group": {
            "_id": None,
            "total": {"$sum": 1},
            "with_comments": {
                "$sum": {
                    "$cond": [
                        {"$isArray": ["$comments"]},
                        1,
                        0
                    ]
                }
            }
        }
    }
]
# Execute the aggregation pipeline and display the result
result = list(movies.aggregate(pipeline_check))
print(result)

# This will tell you how many movies have a comments array and what the distribution of array lengths looks like.

[{'_id': None, 'total': 21349, 'with_comments': 0}]


In [ ]:
# Count movies that have 3 or more comments
pipeline_test = [
    {"$addFields": {"commentCount": {"$size": {"$ifNull": ["$comments", []]}}}},
    {"$match": {"commentCount": {"$gte": 3}}},
    {"$count": "total"}
]
print(list(movies.aggregate(pipeline_test)))

# If this returns a document like { "total": 2 }, then the original pipeline should print those two documents. If it returns an empty list ([]), then indeed there are no movies with three or more comments in the current collection.

[]


In [27]:
pipeline = [
    {
        "$addFields": {
            "commentCount": {"$size": {"$ifNull": ["$comments", []]}}
        }
    },
    {
        "$match": {
            "commentCount": {"$gte": 3}
        }
    },
    {
        "$count": "total"
    }
]

result = movies.aggregate(pipeline)

for doc in result:
    print(f"Number of movies with 3 or more comments: {doc['total']}")